# 06 — ConvLSTM Training

**Week 6 goal:** Train the spatio-temporal model and beat the Week 5 baselines.

By the end of this notebook you will have:
1. Built train / val / test DataLoaders with a temporal validation split
2. Trained a **per-pixel LSTM** (kernel=1 — no spatial context) as an intermediate baseline
3. Trained a **ConvLSTM** (kernel=3 — full spatial context) as the target model
4. Plotted training curves and confirmed both models converged
5. Produced the full benchmark table (persistence / climatology / XGBoost / per-px LSTM / ConvLSTM)
6. Visualised spatial prediction maps and confirmed ConvLSTM predictions are spatially smooth

### ConvLSTM key idea

A regular LSTM has a hidden state of shape `(hidden_size,)` — a vector. **ConvLSTM replaces that vector with a 2D map of shape `(hidden_ch, H, W)`**, and replaces the fully-connected gates with **2D convolutions**.

```
Regular LSTM gate:  sigmoid( W_i · [h_{t-1}, x_t] + b )   W is (hidden, hidden+input)
ConvLSTM gate:      sigmoid( Conv2d([h_{t-1}, x_t]) )       kernel slides over the spatial grid
```

The convolution kernel lets each pixel **share information with its neighbours** at every time step. A drought propagating from the south is visible in the hidden state before it reaches a northern pixel — regular LSTM and XGBoost cannot do this.

### Two experiments

| Model | kernel | What it learns | Expected result |
|-------|--------|----------------|------------------|
| Per-pixel LSTM | 1×1 | Temporal patterns only — same as XGBoost conceptually | ≈ XGBoost |
| ConvLSTM | 3×3 | Temporal + spatial jointly | Should beat per-pixel LSTM |

The only difference is the convolution kernel size — everything else (architecture, training loop, hyperparameters) is identical. This isolates the value of spatial context.

---
**Runtime:** ~30–60 min on Colab T4 GPU. ~5–15 min per-pixel LSTM (much smaller model).  
**Mac local:** MPS (Apple Silicon) is detected automatically and gives ~5× speedup over CPU.

---
## Colab Setup — runs automatically, nothing to skip

Run all cells top to bottom. Colab is detected automatically — Drive is mounted,
latest code is pulled (or cloned on first run), and packages are installed.
Works locally too (all Colab steps are silently skipped).

In [1]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Runs automatically on Google Colab. Silently skipped when running locally.
try:
    from google.colab import drive
    import subprocess, sys, shutil, os
    drive.mount('/content/drive')

    _REPO = '/content/drive/MyDrive/botswana-drought-flood'
    os.makedirs(_REPO, exist_ok=True)

    # Always clone latest code from GitHub into /tmp (avoids Drive git-pull conflicts
    # and stale bytecode caches — Drive is used for data/ only).
    if os.path.exists('/tmp/bdf'):
        shutil.rmtree('/tmp/bdf')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/andrew-simons/botswana-drought-flood',
                    '/tmp/bdf'], check=True, capture_output=True)

    for _f in ['src', 'notebooks', 'scripts', 'app', '.streamlit']:
        _dst = f'{_REPO}/{_f}'
        if os.path.exists(_dst):
            shutil.rmtree(_dst)
        shutil.copytree(f'/tmp/bdf/{_f}', _dst)

    _hash = subprocess.run(['git', 'log', '--oneline', '-1'], cwd='/tmp/bdf',
                           capture_output=True, text=True).stdout.strip()
    print(f'Code updated to: {_hash}')

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rioxarray'], check=True)

    sys.path.insert(0, f'{_REPO}/src')
    CUBE = f'{_REPO}/data/cube'
    print('Ready.')
except ImportError:
    pass  # running locally — CUBE set in cell-setup below

In [ ]:
import sys, json, warnings
warnings.filterwarnings('ignore')

# On Colab: CUBE and sys.path are already set by the Colab Setup cell above.
# Locally: set defaults here.
if 'CUBE' not in globals():
    sys.path.insert(0, '../src')
    CUBE = '../data/cube'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from botswana_ds.data.dataset    import BotswanaCube, compute_norm_stats
from botswana_ds.models.convlstm import ConvLSTMForecaster, drought_weighted_mae_loss
from botswana_ds.train.train     import fit, predict
from botswana_ds.train.metrics   import batch_summary

meta = json.load(open(f'{CUBE}/meta.json'))

T         = meta['time']['T']                  # 282
TRAIN_END = meta['splits']['train_end_idx']    # 228
VAL_START = TRAIN_END - 24                     # 204 — last 2 training years = validation
LABEL_NAMES = meta['label_channels']

INPUT_LEN = 24   # two years of history — captures multi-year drought cycles (El Niño, 2015-16)
HORIZON   = 1    # predict 1 month ahead
BATCH     = 4

# ── Device selection ──────────────────────────────────────────────────────────
# Set DEVICE to 'cpu' to force CPU (e.g. Colab CPU runtime or when GPU quota is gone).
# Set to 'cuda' or 'mps' to force a specific accelerator.
# Leave as None to auto-detect: CUDA → MPS → CPU.
DEVICE = None

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  MPS: {torch.backends.mps.is_available()}')
print(f'CUBE: {CUBE}')
print(f'T={T}  TRAIN_END={TRAIN_END}  VAL_START={VAL_START}')
print(f'Input len={INPUT_LEN}  Horizon={HORIZON}')
print(f'DEVICE={DEVICE!r}  (None = auto-detect)')

---
## Step 1 — DataLoaders

We carve the training period into two parts:
- **Train:** months 0–203 (Jan 2003 – Dec 2018)
- **Val:** months 204–227 (Jan 2019 – Dec 2021) — last 2 years of training period
- **Test:** months 228–281 (Jan 2022 – Jun 2026)

Validation is **temporal** (later in time), not random. A random split would leak future seasonal patterns into training.

The `time_slice` parameter tells BotswanaCube which months' windows to include. We go back `input_len` months before each split boundary so the first window in each set has a full year of context.

In [3]:
# Norm stats computed on training data only (saved in week 4)
import pathlib
norm_path = pathlib.Path(f'{CUBE}/norm_stats.npz')
if norm_path.exists():
    ns = np.load(norm_path)
    norm_stats = {'mean': ns['mean'], 'std': ns['std']}
else:
    norm_stats = compute_norm_stats(CUBE, train_slice=(0, TRAIN_END))
    np.savez(norm_path, **norm_stats)
print('Norm stats loaded.')

# time_slice lower bound goes back input_len so first window has full context
train_ds = BotswanaCube(CUBE, INPUT_LEN, HORIZON, stride=1,
                        time_slice=(0,               VAL_START),  norm_stats=norm_stats)
val_ds   = BotswanaCube(CUBE, INPUT_LEN, HORIZON, stride=1,
                        time_slice=(VAL_START - INPUT_LEN, TRAIN_END), norm_stats=norm_stats)
test_ds  = BotswanaCube(CUBE, INPUT_LEN, HORIZON, stride=1,
                        time_slice=(TRAIN_END - INPUT_LEN, T),    norm_stats=norm_stats)

# pin_memory=False — MPS doesn't support pinned memory (it just prints a warning if True)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=False)

print(f'Train windows: {len(train_ds):4d}  ({len(train_loader)} batches)')
print(f'Val   windows: {len(val_ds):4d}  ({len(val_loader)} batches)')
print(f'Test  windows: {len(test_ds):4d}  ({len(test_loader)} batches)')

sample = train_ds[0]
IN_CH = sample['x'].shape[1]   # C_dyn + 3 label lags — derived from cube, not hardcoded
print(f'\nSample x shape      : {tuple(sample["x"].shape)}  (input_len, C_dyn+3, H, W)')
print(f'Sample y shape      : {tuple(sample["y"].shape)}  (horizon, 3, H, W)')
print(f'IN_CH = {IN_CH}  (use this for all model definitions below)')

Norm stats loaded.
Train windows:  180  (45 batches)
Val   windows:   24  (6 batches)
Test  windows:   24  (6 batches)

Sample x shape      : (24, 12, 182, 188)  (input_len, C_dyn+3, H, W)
Sample y shape      : (1, 3, 182, 188)  (horizon, 3, H, W)
IN_CH = 12  (use this for all model definitions below)


---
## Step 2 — Experiment A: Per-pixel LSTM (kernel=1)

This is a ConvLSTM with a **1×1 kernel** — each pixel only sees itself, never its neighbours. Conceptually equivalent to running a separate LSTM on each pixel's time series independently.

We run this **first** to establish the value of temporal modeling alone, before adding spatial context. If ConvLSTM (kernel=3) doesn't beat this, spatial context is adding nothing.

With `hidden_ch=64` and `kernel=1`, this model has ~21K parameters — still fast to train.

In [ ]:
model_px = ConvLSTMForecaster(
    in_ch=IN_CH,  # C_dyn + 3 label lags — set dynamically from cube shape above
    hidden_ch=64, n_targets=3, horizon=HORIZON,
    kernel=1, static_ch=3,
)

# Residual connection: prediction = head(hidden_state) + label[t-1]
# At init (weights ≈ 0) this is pure persistence; the model learns corrections on top.
print('=== Per-pixel LSTM (kernel=1) ===')
history_px = fit(
    model_px, train_loader, val_loader,
    n_epochs=150, lr=1e-3, patience=25,
    ckpt_path=f'{CUBE}/model_px_lstm.pt',
    device=DEVICE,
    verbose=True,
)

---
## Step 3 — Experiment B: ConvLSTM (kernel=3)

Same architecture, same hyperparameters — only difference is `kernel=3`. The 3×3 convolution lets each pixel exchange information with its 8 immediate neighbours at every time step.

With `hidden_ch=64` and `kernel=3`, this model has ~184K parameters — still small by deep learning standards, but has genuine spatial reasoning capability.

**What to watch:** val loss should drop lower than the per-pixel LSTM, and spatial prediction maps should look smoother.

In [ ]:
# ── Backup original checkpoint before weighted-loss re-train ──────────────────
# Run once before re-training. If the new loss is worse, restore with:
#   shutil.copy(f'{CUBE}/model_convlstm_mae.pt', f'{CUBE}/model_convlstm.pt')
import shutil, pathlib
_orig = pathlib.Path(f'{CUBE}/model_convlstm.pt')
_bak  = pathlib.Path(f'{CUBE}/model_convlstm_mae.pt')
if _orig.exists() and not _bak.exists():
    shutil.copy(_orig, _bak)
    print(f'Backed up → {_bak}')
elif _bak.exists():
    print(f'Backup already exists: {_bak}')
else:
    print('No checkpoint yet — nothing to back up.')


In [ ]:
model_cl = ConvLSTMForecaster(
    in_ch=IN_CH,  # C_dyn + 3 label lags — set dynamically from cube shape above
    hidden_ch=64, n_targets=3, horizon=HORIZON,
    kernel=3, static_ch=3,
)

print('=== ConvLSTM (kernel=3) ===')
history_cl = fit(
    model_cl, train_loader, val_loader,
    n_epochs=150, lr=1e-4,    # lower LR — kernel=3 model trains more stably at 1e-4
    patience=25,
    ckpt_path=f'{CUBE}/model_convlstm.pt',
    device=DEVICE,
    verbose=True,
    loss_fn=drought_weighted_mae_loss,
)

---
## Step 4 — Training curves

A healthy training curve shows:
- Train loss and val loss both decreasing early
- Val loss levelling off before train loss (expected — training set is larger)
- No large gap between train and val (large gap = overfitting)
- Early stopping fires before the model degrades

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, history, title in [
    (axes[0], history_px, 'Per-pixel LSTM (kernel=1)'),
    (axes[1], history_cl, 'ConvLSTM (kernel=3)'),
]:
    ax.plot(history['train'], label='train MAE')
    ax.plot(history['val'],   label='val MAE')
    best_epoch = int(np.argmin(history['val']))
    ax.axvline(best_epoch, color='red', ls='--', lw=1, label=f'best epoch {best_epoch+1}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Masked MAE')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CUBE}/06_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Per-pixel LSTM  best val MAE = {min(history_px["val"]):.4f}  (epoch {np.argmin(history_px["val"])+1})')
print(f'ConvLSTM        best val MAE = {min(history_cl["val"]):.4f}  (epoch {np.argmin(history_cl["val"])+1})')

---
## Step 5 — Test set evaluation

Load the best checkpoint for each model and evaluate on the 54 test months (Jan 2022 – Jun 2026).
We report the same metrics as Week 5 so results are directly comparable.

In [ ]:
import importlib
import botswana_ds.train.train as _train_mod
importlib.reload(_train_mod)
from botswana_ds.train.train import predict

mask = np.load(f'{CUBE}/mask.npy')  # (H, W)

def eval_model(model, ckpt_path, loader, label_names, label):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    model.load_state_dict(ckpt['state_dict'])
    preds, targets, _ = predict(model, loader, device='cpu')
    # preds/targets: (N, horizon, 3, H, W) — squeeze horizon=1
    preds   = preds[:, 0]    # (N, 3, H, W)
    targets = targets[:, 0]  # (N, 3, H, W)
    metrics = {}
    for c, name in enumerate(label_names):
        metrics[name] = batch_summary(preds[:, c], targets[:, c], mask)
    return metrics

print('Evaluating per-pixel LSTM...')
px_metrics = eval_model(model_px, f'{CUBE}/model_px_lstm.pt',
                        test_loader, LABEL_NAMES, 'px')

print('Evaluating ConvLSTM...')
cl_metrics = eval_model(model_cl, f'{CUBE}/model_convlstm.pt',
                        test_loader, LABEL_NAMES, 'cl')

# ── Load week-5 baseline numbers ──────────────────────────────────────────
# Re-run persistence + climatology quickly to get the same metrics object
labels = np.load(f'{CUBE}/labels.npy', mmap_mode='r')
N_TEST = T - TRAIN_END

persist_preds   = np.asarray(labels[TRAIN_END - 1 : T - 1])
persist_targets = np.asarray(labels[TRAIN_END : T])
persist_metrics = {name: batch_summary(persist_preds[:, c], persist_targets[:, c], mask)
                   for c, name in enumerate(LABEL_NAMES)}

clim = np.full((12, 3, *mask.shape), np.nan, dtype=np.float32)
for m in range(12):
    idx = [t for t in range(TRAIN_END) if t % 12 == m]
    clim[m] = np.nanmean(labels[idx], axis=0)
clim_preds   = np.stack([clim[(TRAIN_END + i) % 12] for i in range(N_TEST)])
clim_metrics = {name: batch_summary(clim_preds[:, c], persist_targets[:, c], mask)
                for c, name in enumerate(LABEL_NAMES)}

In [ ]:
import pathlib, json as _json

# Load XGBoost metrics saved by notebook 05 (avoids re-running XGBoost here,
# which would load ~1.2 GB dynamic cube into RAM and crash the kernel).
_xgb_path = pathlib.Path(f'{CUBE}/xgb_metrics.json')
if _xgb_path.exists():
    xgb_metrics = _json.load(open(_xgb_path))
    print('XGBoost metrics loaded from file.')
else:
    xgb_metrics = None
    print('XGBoost metrics not found — run notebook 05 first, then re-run this cell.')

# ── Full benchmark table ──────────────────────────────────────────────────
all_models = [
    ('Persistence',    persist_metrics),
    ('Climatology',    clim_metrics),
]
if xgb_metrics:
    all_models.append(('XGBoost', xgb_metrics))
all_models += [
    ('Per-px LSTM',  px_metrics),
    ('ConvLSTM',     cl_metrics),
]

rows = []
for model_name, mdict in all_models:
    for label in LABEL_NAMES:
        m = mdict[label]
        rows.append({'Model': model_name, 'Label': label,
                     'MAE':  round(m['mae'],  3),
                     'RMSE': round(m['rmse'], 3),
                     'R²':   round(m['r2'],   3),
                     'F1':   round(m['f1'],   3)})

df = pd.DataFrame(rows)
print('\n=== FULL BENCHMARK TABLE ===')
print(df.to_string(index=False))
print('\n── MAE pivot ──')
print(df.pivot_table('MAE', index='Model', columns='Label').to_string())
print('\n── F1 pivot ──')
print(df.pivot_table('F1',  index='Model', columns='Label').to_string())

---
## Step 6 — Spatial prediction maps

The key visual test: **ConvLSTM predictions should look spatially smooth** — drought patches, not random per-pixel noise.

XGBoost has no spatial reasoning so its predictions often look salt-and-pepper. ConvLSTM should look more like the ground truth.

In [ ]:
VIS_MONTH = 11   # 0=Jan2022 … 11=Dec2022 — change to explore
LABEL_IDX = 0    # 0=SPI-3  1=NDVI-anom  2=SM-anom

# Reload best ConvLSTM predictions
ckpt = torch.load(f'{CUBE}/model_convlstm.pt', map_location='cpu', weights_only=True)
model_cl.load_state_dict(ckpt['state_dict'])
cl_preds_all, cl_targets_all, _ = predict(model_cl, test_loader, device='cpu')
cl_preds_all = cl_preds_all[:, 0]    # (24, 3, H, W)
cl_targets_all = cl_targets_all[:, 0]

ckpt_px = torch.load(f'{CUBE}/model_px_lstm.pt', map_location='cpu', weights_only=True)
model_px.load_state_dict(ckpt_px['state_dict'])
px_preds_all, _, _ = predict(model_px, test_loader, device='cpu')
px_preds_all = px_preds_all[:, 0]   # (24, 3, H, W)

# Ground truth is the same for all models
ground_truth = np.where(mask, cl_targets_all[VIS_MONTH, LABEL_IDX], np.nan)

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
abs_m = TRAIN_END + VIS_MONTH
title_month = f'{month_names[abs_m % 12]} {2003 + abs_m // 12}'

to_plot = [
    (ground_truth, 'Ground Truth'),
    (np.where(mask, px_preds_all[VIS_MONTH, LABEL_IDX], np.nan), 'Per-pixel LSTM'),
    (np.where(mask, cl_preds_all[VIS_MONTH, LABEL_IDX], np.nan), 'ConvLSTM'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (data, name) in zip(axes, to_plot):
    im = ax.imshow(data, cmap='RdYlBu', vmin=-2.5, vmax=2.5, origin='upper')
    ax.set_title(name, fontsize=10)
    ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.7, label=f'{LABEL_NAMES[LABEL_IDX]} (z-score)')
fig.suptitle(f'{title_month} — {LABEL_NAMES[LABEL_IDX]}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CUBE}/06_prediction_maps.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/cube/06_prediction_maps.png')

---
## Step 7 — Monthly MAE time series

Same plot as Week 5 with the two new models added. Watch for:
- ConvLSTM should track below per-pixel LSTM on most months
- Error spikes in Nov–Feb (rainy season) are expected — that's when conditions change fastest

In [ ]:
valid_h, valid_w = np.where(mask)

def monthly_mae_series(preds_3d, targets_3d):
    """preds_3d / targets_3d: (N_TEST, H, W) for one label channel."""
    maes = []
    for t in range(N_TEST):
        p = preds_3d[t, valid_h, valid_w]
        g = targets_3d[t, valid_h, valid_w]
        ok = ~np.isnan(p) & ~np.isnan(g)
        maes.append(float(np.mean(np.abs(p[ok] - g[ok]))) if ok.any() else np.nan)
    return maes

mn = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
x_labels = [f"{mn[(TRAIN_END+i)%12]}\n{2003+(TRAIN_END+i)//12}" for i in range(N_TEST)]

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
gt_3d = cl_targets_all   # (54, 3, H, W)

for c, (name, ax) in enumerate(zip(LABEL_NAMES, axes)):
    ax.plot(monthly_mae_series(persist_preds[:, c], persist_targets[:, c]),
            'o-', label='Persistence', ms=4, alpha=0.7)
    ax.plot(monthly_mae_series(px_preds_all[:, c],  gt_3d[:, c]),
            's-', label='Per-px LSTM', ms=4)
    ax.plot(monthly_mae_series(cl_preds_all[:, c],  gt_3d[:, c]),
            '^-', label='ConvLSTM',    ms=4, lw=2)
    ax.set_ylabel(f'{name}\nMAE', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xticks(range(N_TEST))

axes[-1].set_xticklabels(x_labels, fontsize=7)
fig.suptitle('Monthly MAE — test period Jan 2022 – Jun 2026', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CUBE}/06_mae_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Week 6 success criteria

| Check | Expected |
|-------|----------|
| Both models converged (val loss decreasing early) | ✓ — check training curves |
| Per-pixel LSTM val MAE < Persistence MAE | ✓ — temporal modeling helps |
| ConvLSTM val MAE < Per-pixel LSTM val MAE | ✓ — spatial context helps |
| ConvLSTM SPI-3 MAE < 0.536 (beats XGBoost) | target |
| ConvLSTM NDVI MAE < 0.466 (beats persistence) | target |
| ConvLSTM spatial maps look smoother than per-pixel LSTM | ✓ — visible in plot |
| Checkpoints saved to `data/cube/` | ✓ |

### If ConvLSTM doesn't beat the baselines

Common causes and fixes:
- **Train longer:** increase `n_epochs=150`, `patience=20`
- **More capacity:** try `hidden_ch=64` (4× parameters)
- **More history:** try `INPUT_LEN=24` (2 years) — captures multi-year drought cycles
- **Lower lr:** try `lr=3e-4` if loss is noisy

**All checks pass? Ready for Week 7** — SHAP/Integrated Gradients explanations + benchmark write-up.